# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [13]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Respuesta:**
- **Gymnasium** proporciona el ambiente ya implementado: FrozenLake, sus estados, acciones, recompensas y transiciones.
- El agente no está definido como una clase separada; está representado por la **Q-table** y las funciones `epsilon_greedy_policy`, `greedy_policy` y `train_q_learning`.
- `train_q_learning` actualiza la Q-table usando las experiencias obtenidas al interactuar con el ambiente.

## 2. Crear FrozenLake


In [14]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

**Respuesta:**
1. **16 filas**, una por cada estado de FrozenLake (`env.observation_space.n = 16`).
2. **4 columnas**, una por cada acción posible (`env.action_space.n = 4`).
3. Cada celda \(Q[s,a]\) estima qué tan conveniente es ejecutar la acción \(a\) cuando el agente está en el estado \(s\), según la recompensa futura esperada.

La forma de la tabla es **(16, 4)**.

In [15]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

**Respuesta:**
- Que todos los valores sean cero **no significa que todas las acciones sean malas**; significa que el agente todavía no ha aprendido qué tan buenas son.
- Si varias acciones tienen el mismo valor máximo, `greedy_policy` elige aleatoriamente entre las acciones empatadas. Al inicio, como todos los valores son cero, cualquiera puede ser elegida.

## 3. Ejecutar una transición


In [16]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 2
reward     = 0
next_state = 1
done       = False


### 💬 Actividad 3 — Identificar la experiencia

La experiencia tiene la forma \((s,a,r,s')\): estado actual, acción elegida, recompensa recibida y estado siguiente.

**Respuesta:** sustituye los espacios por los valores que imprimió la celda anterior:

\[
(\text{state},\ \text{action},\ \text{reward},\ \text{next\_state})
\]

En Value Iteration se trabaja con el modelo conocido del ambiente; no se necesita obtener cada transición mediante una interacción real. En Q-Learning, la experiencia observada incluye la **recompensa \(r\)** y el **estado siguiente \(s'\)** de esa interacción.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [17]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [18]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [19]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [20]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

**Respuesta:**
- En `Q_initial`, todas las estimaciones empiezan en cero, por lo que no hay una preferencia aprendida.
- En `Q_early`, las experiencias de los primeros 50 episodios ya modificaron algunos valores, pero el aprendizaje puede estar incompleto.
- En `Q_trained`, tras 10 000 episodios, los valores Q suelen representar mejor qué acciones llevan a la meta y permiten evitar los agujeros.
- El ambiente y las acciones no cambian; lo que cambia es la **Q-table**, que guía las decisiones del agente.

### 💬 Actividad 4 — Leer una fila de Q

Ejecuta la celda de código añadida debajo para mostrar los cuatro valores de un estado y la acción greedy correspondiente.

**Interpretación:** la acción elegida por \(rg\max_a Q(s,a)\) es la que tiene el valor Q más alto en ese estado. Si hay empate, el código puede escoger cualquiera de las acciones empatadas.

In [21]:
# Actividad 4: elige un estado (0 a 15) y muestra sus valores Q\n
s = 0\n
print('Estado seleccionado:', s)\n
print('Left :', Q_trained[s, 0])\n
print('Down :', Q_trained[s, 1])\n
print('Right:', Q_trained[s, 2])\n
print('Up   :', Q_trained[s, 3])\n
best_actions = np.flatnonzero(Q_trained[s] == np.max(Q_trained[s]))\n
print('Acción greedy (código):', int(np.random.choice(best_actions)))

SyntaxError: unexpected character after line continuation character (3783590978.py, line 2)

## 5. Evaluar la política aprendida


In [ ]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


### 💬 Actividad 5 — Exploration vs. exploitation

**Conclusión:**

Durante el entrenamiento usamos \(\epsilon\)-greedy para explorar acciones nuevas y aprender del ambiente, además de aprovechar lo aprendido. Durante la evaluación no exploramos porque queremos medir el desempeño de la política aprendida, eligiendo la acción con mayor valor Q. Las acciones aleatorias introducirían variabilidad y podrían empeorar la recompensa medida.

## 6. Experimento: FrozenLake estocástico


In [ ]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

### 💬 Actividad 6 — Determinístico vs. estocástico

**Discusión:**
- **Ambiente:** con `is_slippery=False`, la acción produce la transición prevista de forma determinística. Con `is_slippery=True`, el hielo puede hacer que el agente se deslice y termine en un estado distinto al esperado; las transiciones son probabilísticas.
- **Algoritmo:** la ecuación de actualización de Q-Learning no cambia:

\[
Q(s,a) \leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right]
\]

  Lo que cambia es la experiencia observada \((s,a,r,s')\) y, por tanto, los valores aprendidos. En el ambiente estocástico normalmente se necesita más experiencia y la recompensa puede ser menos consistente.

# Cierre de clase

**1. ¿Qué almacena \(Q(s,a)\)?**

> Una estimación de la recompensa futura acumulada esperada al ejecutar la acción \(a\) desde el estado \(s\) y continuar siguiendo una política.

**2. ¿De dónde sale \(\max_{a'}Q(s',a')\)?**

> De la fila de la Q-table correspondiente al estado siguiente \(s'\). Es el mayor valor Q entre todas las acciones posibles desde ese estado y estima la mejor continuación futura.

**3. ¿Por qué necesitamos \(\epsilon\)-greedy?**

> Para equilibrar exploración y explotación: con probabilidad \(\epsilon\) se prueba una acción aleatoria y, en caso contrario, se elige una acción con el mayor valor Q conocido. Así el agente puede descubrir mejores caminos sin dejar de aprovechar lo aprendido.

**4. ¿Por qué Q-Learning es model-free?**

> Porque no necesita conocer previamente las probabilidades de transición ni la función completa de recompensas del ambiente. Aprende los valores Q a partir de las experiencias obtenidas al interactuar con él.